# Imports and Configurations

In [ ]:
# Experiment
EXP_NUM = 1
DATASET = 'IDSM'
CONTENT = 'dry/wet'
GRANULARITY = 2
TARGET_COL = 'category'
CLASS_MAP = {
    'no-rain': 'Dry', 
    'light': 'Wet', 'moderate': 'Wet', 
    'heavy': 'Wet', 'violent': 'Wet'
}

import pandas as pd
import sys
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

current_path = Path.cwd()
PROJECT_ROOT = None

for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Project root added to sys.path")
else:
    print("ERROR: Unable to find the project root.")

from rainfall_acoustic_classification.visualization import VisualizationEngine
from rainfall_acoustic_classification.utils import get_standard_logger

# Feature Engineering
from rainfall_acoustic_classification.feature_engineering import (
    # Core modules
    ExperimentCreator, ExperimentConfig, 
    
    # Single Metric (PSD_mean) 
    SingleSelectorConfig, build_single_selector,

    # Metrics Selection
    VectorSelectorConfig, VectorSelector,
    
    # Vizualization
    plot_correlation_heatmap, plot_individual_boxplots, plot_feature_importance, plot_fisher_scores
)

# Modeling
from rainfall_acoustic_classification.modeling import (
    # Classification
    ClassifierFactory, ClassifierConfig,

    # Hyperparameter Optimization
    ModelOptimizer,TuningConfig,

    # Validation
    ModelEvaluator, ValidationConfig,

    #Vizualization
    plot_confusion_matrix_grid, plot_multiclass_pr_curve, plot_experiment_performance_heatmap
)

# Directories
DATA_DIR = PROJECT_ROOT / "data" / "processed" / DATASET
REPORTS_DIR = PROJECT_ROOT / "reports" / "vector_selector" / "experiment_3" / DATASET
MODELS_DIR = PROJECT_ROOT / "models" / "vector_selector" / "experiment_3" / DATASET

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Data Loading and Mapping

In [ ]:
df_train_metrics = pd.read_csv(DATA_DIR /  f"{DATASET}_train_metrics.csv")
df_val_metrics = pd.read_csv(DATA_DIR / f"{DATASET}_val_metrics.csv")
df_test_metrics = pd.read_csv(DATA_DIR / f"{DATASET}_test_metrics.csv")

print(f"Raw Shapes -> Train: {df_train_metrics.shape}, Val: {df_val_metrics.shape}, Test: {df_test_metrics.shape}")

In [ ]:
metadata_cols = [
    'file_name', 'timestamp', 'period', 'mm_5min', 'mm_hr', 
    'category', 'recorder', 'location', 'file_path', 'extension', 
    'year', 'month', 'day', 'hour', 'minute', 'second', 'wet', 'split', 
    'should_augment', 'segment_idx', 'offset_sec', 'aug_params'
    ]

exp_config = ExperimentConfig(content=CONTENT,
                              granularity=GRANULARITY, 
                              label_col=TARGET_COL,
                              metadata_cols = metadata_cols,
                              custom_mapping=CLASS_MAP)

X_train_metrics, y_train_metrics, _ = ExperimentCreator.extract_X_y_meta(df_train_metrics, config=exp_config)
X_train, y_train = ExperimentCreator.apply_experiment_rules(X_train_metrics, y_train_metrics, config=exp_config)


X_val_metrics, y_val_metrics, _ = ExperimentCreator.extract_X_y_meta(df_val_metrics, config=exp_config)
X_val, y_val = ExperimentCreator.apply_experiment_rules(X_val_metrics, y_val_metrics, config=exp_config)


X_test_metrics, y_test_metrics, _ = ExperimentCreator.extract_X_y_meta(df_test_metrics, config=exp_config)
X_test, y_test = ExperimentCreator.apply_experiment_rules(X_test_metrics, y_test_metrics, config=exp_config)

print(f"Classes post mapping -> Train: {y_train.unique()}, Val: {y_val.unique()}, Test: {y_test.unique()}")

# Feature Engineering (Metrics Selection and PSD)

In [ ]:
metrics_selector_config = VectorSelectorConfig(corr_threshold=0.85, 
                                               rf_n_estimators=100, 
                                               fisher_percentile=10)
metrics_selector = VectorSelector(metrics_selector_config)

X_train_selected_metrics = metrics_selector.fit_transform(X_train, y_train)

survivors = metrics_selector.get_feature_names_out()
print(f"Original Features: {X_train.shape[1]} -> Selected: {X_train_selected_metrics.shape[1]}")
print(f"Selected ones: {survivors}")

X_val_selected_metrics = metrics_selector.transform(X_val)
X_test_selected_metrics = metrics_selector.transform(X_test)

df_feature_tracking = metrics_selector.get_metrics_report()

## Graphics

In [ ]:
display(df_feature_tracking.head(50))

In [ ]:
viz_engine = VisualizationEngine()
palette = viz_engine.get_rain_color_palette()
rain_palette = dict(list(palette.items())[5:7])
exp_palette = {k: v for k, v in rain_palette.items() if k in y_train.unique()}
ordered_classes = list(exp_palette.keys())

selected_metrics = df_feature_tracking[df_feature_tracking['Survived_Pipeline'] == 1]['Feature_Name'].tolist()
X_train_fischer_metrics = X_train[selected_metrics]

In [ ]:
plot_correlation_heatmap(
    df=X_train_fischer_metrics,
    threshold=0.85,
    title=f"Selected Metrics Correlation"
)

In [ ]:
plot_individual_boxplots(
    X=X_train_fischer_metrics, 
    y=y_train, 
    color_map=rain_palette,
    class_order=ordered_classes
)

In [ ]:
plot_fisher_scores(
    df_tracking=df_feature_tracking, 
    top_n=10,
    title="Fisher Score"
)

## PSD

In [ ]:
psd_selector_config = SingleSelectorConfig(target_feature='psd_mean', 
                                           fisher_percentile=60)
psd_selector = build_single_selector(psd_selector_config)


X_train_psd = psd_selector.fit_transform(X_train, y_train)
print(f"Original Features: {X_train.shape[1]} -> Selected: {X_train_psd.shape[1]}")
print(f"Selected ones: {X_train_psd.columns}")

X_val_psd = psd_selector.transform(X_val)
X_test_psd = psd_selector.transform(X_test)

In [ ]:
plot_individual_boxplots(
    X=X_train_psd, 
    y=y_train, 
    color_map=rain_palette, 
    class_order=ordered_classes
)

# Modeling

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Instancia o codificador (tradutor de Texto para Número)
le = LabelEncoder()

# Transforma os textos ('Heavy', 'Light') em inteiros (0, 1, 2, 3, 4)
# e preserva o índice original do Pandas
y_train_enc = pd.Series(le.fit_transform(y_train), index=y_train.index, name=TARGET_COL)
y_val_enc   = pd.Series(le.transform(y_val), index=y_val.index, name=TARGET_COL)
y_test_enc  = pd.Series(le.transform(y_test), index=y_test.index, name=TARGET_COL)

print("✅ Variáveis 'y_train_enc', 'y_val_enc' e 'y_test_enc' recriadas e salvas na memória!")
print(f"Mapeamento interno: {dict(zip(le.classes_, le.transform(le.classes_)))}")

## PSD

In [ ]:
sgd_config = ClassifierConfig(model_type='sgd', random_state=42, n_jobs=-1)
sgd_base = ClassifierFactory.build(sgd_config)

sgd_grid = {
    'alpha': [0.0001, 0.001],
    'penalty': ['l2', 'elasticnet'],
    'class_weight': ['balanced']
}

tuning_config = TuningConfig(param_grid=sgd_grid, scoring_metric='f1_macro', n_jobs=-1)

sgd_champion = ModelOptimizer.optimize(
    estimator=sgd_base, 
    X_train=X_train_psd, 
    y_train=y_train, 
    X_val=X_val_psd, 
    y_val=y_val, 
    config=tuning_config
)

val_config = ValidationConfig(average_method='macro', return_report_dict=True)

metrics_sgd = ModelEvaluator.evaluate(
    model=sgd_champion, 
    X_test=X_test_psd, 
    y_test=y_test, 
    config=val_config
)

for chave, valor in metrics_sgd.items():
    if isinstance(valor, (int, float, str)):
        print(f" -> {chave}: {valor}")

In [ ]:
lr_config = ClassifierConfig(model_type='lr', random_state=42)
lr_base = ClassifierFactory.build(lr_config)

lr_grid = {
    'C': [0.1, 1.0, 10.0], 
    'class_weight': ['balanced'], 
    'solver': ['lbfgs'], 
    'max_iter': [1000]
}

tuning_config_lr = TuningConfig(param_grid=lr_grid, scoring_metric='f1_macro', n_jobs=-1)

lr_champion = ModelOptimizer.optimize(
    estimator=lr_base, 
    X_train=X_train_psd, 
    y_train=y_train_enc,
    X_val=X_val_psd, 
    y_val=y_val_enc, 
    config=tuning_config_lr
)

val_config = ValidationConfig(average_method='macro', return_report_dict=True)
metrics_lr = ModelEvaluator.evaluate(
    model=lr_champion, 
    X_test=X_test_psd, 
    y_test=y_test_enc, 
    config=val_config
)

for chave, valor in metrics_lr.items():
    if isinstance(valor, (int, float, str)):
        print(f" -> {chave}: {valor}")


In [ ]:
rf_config = ClassifierConfig(model_type='rf', random_state=42)
rf_base = ClassifierFactory.build(rf_config)

rf_grid = {
    'n_estimators': [100, 200], 
    'max_depth': [None, 5, 10], 
    'min_samples_leaf': [1, 2]
}
tuning_config_rf = TuningConfig(param_grid=rf_grid, scoring_metric='f1_macro', n_jobs=-1)

rf_champion = ModelOptimizer.optimize(
    estimator=rf_base, 
    X_train=X_train_psd, 
    y_train=y_train_enc, 
    X_val=X_val_psd, 
    y_val=y_val_enc, 
    config=tuning_config_rf
)

metrics_rf = ModelEvaluator.evaluate(
    model=rf_champion, 
    X_test=X_test_psd, 
    y_test=y_test_enc, 
    config=val_config
)

for chave, valor in metrics_rf.items():
    if isinstance(valor, (int, float, str)):
        print(f" -> {chave}: {valor}")

In [ ]:
xgb_config = ClassifierConfig(model_type='xgb', random_state=42)
xgb_base = ClassifierFactory.build(xgb_config)

xgb_grid = {
    'n_estimators': [100, 200], 
    'max_depth': [3, 5], 
    'learning_rate': [0.01, 0.1]
}
tuning_config_xgb = TuningConfig(param_grid=xgb_grid, scoring_metric='f1_macro', n_jobs=-1)

xgb_champion = ModelOptimizer.optimize(
    estimator=xgb_base, 
    X_train=X_train_psd, 
    y_train=y_train_enc, # OBRIGATÓRIO: Variável Numérica (0 a 4)
    X_val=X_val_psd, 
    y_val=y_val_enc,     # OBRIGATÓRIO: Variável Numérica (0 a 4)
    config=tuning_config_xgb
)

metrics_xgb = ModelEvaluator.evaluate(
    model=xgb_champion, 
    X_test=X_test_psd, 
    y_test=y_test_enc,   # OBRIGATÓRIO: Variável Numérica (0 a 4)
    config=val_config
)

for chave, valor in metrics_xgb.items():
    if isinstance(valor, (int, float, str)):
        print(f" -> {chave}: {valor}")

## Selected Metrics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

MAX_FEATURES = 10
top_10_features = df_feature_tracking[df_feature_tracking['Survived_Pipeline'] == 1]['Feature_Name'].head(MAX_FEATURES).tolist()

In [ ]:
sgd_config = ClassifierConfig(model_type='sgd', random_state=42)
sgd_grid = {
    'alpha': [0.0001, 0.001], 
    'penalty': ['l2', 'elasticnet'], 
    'class_weight': ['balanced']
}
tuning_config_sgd = TuningConfig(param_grid=sgd_grid, scoring_metric='f1_macro', n_jobs=-1)
val_config = ValidationConfig(average_method='macro', return_report_dict=False)

incremental_results_sgd = []

for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_adicionada = current_features[-1]
    
    print(f"--- Treinando com {k} Feature(s) [Adicionada agora: {feature_adicionada}] ---")
    
    # Fatiamento da matriz usando as listas de nomes
    X_tr_slice = X_train_selected_metrics[current_features]
    X_va_slice = X_val_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        # A. Constrói o modelo
        sgd_base = ClassifierFactory.build(sgd_config)
        
        # B. Otimiza (Tuning)
        sgd_champion = ModelOptimizer.optimize(
            estimator=sgd_base, 
            X_train=X_tr_slice, 
            y_train=y_train_enc, # Usando a variável numérica que criamos para evitar erros
            X_val=X_va_slice, 
            y_val=y_val_enc, 
            config=tuning_config_sgd
        )
        
        # C. Avalia no Teste (O veredito final daquele conjunto)
        metrics_sgd = ModelEvaluator.evaluate(
            model=sgd_champion, 
            X_test=X_te_slice, 
            y_test=y_test_enc, 
            config=val_config
        )
        
        # D. Extração Segura das Métricas (Tratando possíveis mudanças no dicionário)
        f1  = metrics_sgd.get('f1_macro', metrics_sgd.get('f1_score', metrics_sgd.get('f1', 0.0)))
        acc = metrics_sgd.get('accuracy', metrics_sgd.get('accuracy_score', 0.0))
        
        incremental_results_sgd.append({
            'Dimensao': k,
            'Ultima_Feature_Inclusa': feature_adicionada,
            'F1_Macro': f1,
            'Accuracy': acc
        })
        
    except Exception as e:
        print(f"❌ Falha no passo com {k} features: {e}")

# 4. Tabela de Resultados
df_sgd_incremental = pd.DataFrame(incremental_results_sgd)
print("\n📊 Tabela de Desempenho Incremental:")
display(df_sgd_incremental)

# 5. O Gráfico Definitivo (Curva de Aprendizado por Dimensão)
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_sgd_incremental, 
    x='Dimensao', 
    y='F1_Macro', 
    marker='o', 
    color='purple', 
    linewidth=2.5,
    markersize=8
)

plt.title("Impacto da Adição Incremental de Features Acústicas (SGD)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Quantidade de Features Utilizadas (Top Fisher Score)", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)
plt.xticks(range(1, MAX_FEATURES + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# Salva se a pasta REPORTS_DIR estiver definida
if 'REPORTS_DIR' in locals():
    plt.savefig(REPORTS_DIR / f"{DATASET}_E{EXP_NUM}_SGD_Incremental.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# ==============================================================================
# Teste Unitário de Dimensão: Incremental Feature Scaling no LR
# ==============================================================================
print("="*60)
print("🚀 Teste de Escalonamento de Features (Incremental) - Regressão Logística (LR)")
print("="*60)

# Configurações do LR
lr_config = ClassifierConfig(model_type='lr', random_state=42)
lr_grid = {
    'C': [0.1, 1.0, 10.0], 
    'class_weight': ['balanced'], 
    'solver': ['lbfgs'], 
    'max_iter': [1000]
}
tuning_config_lr = TuningConfig(param_grid=lr_grid, scoring_metric='f1_macro', n_jobs=-1)

incremental_results_lr = []

for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_adicionada = current_features[-1]
    print(f"--- Treinando LR com {k} Feature(s) [+ {feature_adicionada}] ---")
    
    X_tr_slice = X_train_selected_metrics[current_features]
    X_va_slice = X_val_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        lr_base = ClassifierFactory.build(lr_config)
        lr_champion = ModelOptimizer.optimize(
            estimator=lr_base, X_train=X_tr_slice, y_train=y_train_enc, 
            X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_lr
        )
        metrics_lr = ModelEvaluator.evaluate(model=lr_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
        
        f1  = metrics_lr.get('f1_macro', metrics_lr.get('f1_score', metrics_lr.get('f1', 0.0)))
        acc = metrics_lr.get('accuracy', metrics_lr.get('accuracy_score', 0.0))
        
        incremental_results_lr.append({'Dimensao': k, 'Ultima_Feature_Inclusa': feature_adicionada, 'F1_Macro': f1, 'Accuracy': acc})
    except Exception as e:
        print(f"❌ Falha no passo com {k} features: {e}")

# Gráfico
df_lr_incremental = pd.DataFrame(incremental_results_lr)
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_lr_incremental, x='Dimensao', y='F1_Macro', marker='s', color='blue', linewidth=2.5, markersize=8)
plt.title("Impacto da Adição Incremental de Features Acústicas (Logistic Regression)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Quantidade de Features Utilizadas (Top Fisher Score)", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)
plt.xticks(range(1, MAX_FEATURES + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Teste Unitário de Dimensão: Incremental Feature Scaling no RF
# ==============================================================================
print("="*60)
print("🚀 Teste de Escalonamento de Features (Incremental) - Random Forest (RF)")
print("="*60)

# Configurações do RF
rf_config = ClassifierConfig(model_type='rf', random_state=42)
rf_grid = {
    'n_estimators': [100, 200], 
    'max_depth': [None, 5, 10], 
    'min_samples_leaf': [1, 2]
}
tuning_config_rf = TuningConfig(param_grid=rf_grid, scoring_metric='f1_macro', n_jobs=-1)

incremental_results_rf = []

for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_adicionada = current_features[-1]
    print(f"--- Treinando RF com {k} Feature(s) [+ {feature_adicionada}] ---")
    
    X_tr_slice = X_train_selected_metrics[current_features]
    X_va_slice = X_val_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        rf_base = ClassifierFactory.build(rf_config)
        rf_champion = ModelOptimizer.optimize(
            estimator=rf_base, X_train=X_tr_slice, y_train=y_train_enc, 
            X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_rf
        )
        metrics_rf = ModelEvaluator.evaluate(model=rf_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
        
        f1  = metrics_rf.get('f1_macro', metrics_rf.get('f1_score', metrics_rf.get('f1', 0.0)))
        acc = metrics_rf.get('accuracy', metrics_rf.get('accuracy_score', 0.0))
        
        incremental_results_rf.append({'Dimensao': k, 'Ultima_Feature_Inclusa': feature_adicionada, 'F1_Macro': f1, 'Accuracy': acc})
    except Exception as e:
        print(f"❌ Falha no passo com {k} features: {e}")

# Gráfico
df_rf_incremental = pd.DataFrame(incremental_results_rf)
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_rf_incremental, x='Dimensao', y='F1_Macro', marker='D', color='green', linewidth=2.5, markersize=8)
plt.title("Impacto da Adição Incremental de Features Acústicas (Random Forest)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Quantidade de Features Utilizadas (Top Fisher Score)", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)
plt.xticks(range(1, MAX_FEATURES + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Teste Unitário de Dimensão: Incremental Feature Scaling no XGBoost
# ==============================================================================
print("="*60)
print("🚀 Teste de Escalonamento de Features (Incremental) - XGBoost (XGB)")
print("="*60)

# Configurações do XGBoost
xgb_config = ClassifierConfig(model_type='xgb', random_state=42)
xgb_grid = {
    'n_estimators': [100, 200], 
    'max_depth': [3, 5], 
    'learning_rate': [0.01, 0.1]
}
tuning_config_xgb = TuningConfig(param_grid=xgb_grid, scoring_metric='f1_macro', n_jobs=-1)

incremental_results_xgb = []

for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_adicionada = current_features[-1]
    print(f"--- Treinando XGB com {k} Feature(s) [+ {feature_adicionada}] ---")
    
    X_tr_slice = X_train_selected_metrics[current_features]
    X_va_slice = X_val_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        xgb_base = ClassifierFactory.build(xgb_config)
        xgb_champion = ModelOptimizer.optimize(
            estimator=xgb_base, X_train=X_tr_slice, y_train=y_train_enc, 
            X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_xgb
        )
        metrics_xgb = ModelEvaluator.evaluate(model=xgb_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
        
        f1  = metrics_xgb.get('f1_macro', metrics_xgb.get('f1_score', metrics_xgb.get('f1', 0.0)))
        acc = metrics_xgb.get('accuracy', metrics_xgb.get('accuracy_score', 0.0))
        
        incremental_results_xgb.append({'Dimensao': k, 'Ultima_Feature_Inclusa': feature_adicionada, 'F1_Macro': f1, 'Accuracy': acc})
    except Exception as e:
        print(f"❌ Falha no passo com {k} features: {e}")

# Gráfico
df_xgb_incremental = pd.DataFrame(incremental_results_xgb)
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_xgb_incremental, x='Dimensao', y='F1_Macro', marker='^', color='orange', linewidth=2.5, markersize=8)
plt.title("Impacto da Adição Incremental de Features Acústicas (XGBoost)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Quantidade de Features Utilizadas (Top Fisher Score)", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)
plt.xticks(range(1, MAX_FEATURES + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Teste Unitário de Dimensão: Incremental Feature Scaling no Modelo Master (NuSVC Poly)
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rainfall_acoustic_classification.modeling.classifiers import ClassifierFactory, ClassifierConfig
from rainfall_acoustic_classification.modeling.tuning import ModelOptimizer, TuningConfig
from rainfall_acoustic_classification.modeling.validation import ModelEvaluator, ValidationConfig

print("="*60)
print("🚀 Teste de Escalonamento (Incremental) - MASTER MODEL (NuSVC Poly)")
print("="*60)

MAX_FEATURES = 10

# 1. Configurações TRAVADAS no seu Campeão Absoluto
# Isso acelera o teste em 90x, pois não faremos busca cega, apenas confirmaremos a dimensão.
nusvc_config = ClassifierConfig(model_type='nusvc', random_state=42)

nusvc_master_grid = {
    'kernel': ['poly'],
    'nu': [0.5],
    'degree': [3],
    'coef0': [1.0],
    'gamma': ['scale'],
    'class_weight': ['balanced']
}

tuning_config_master = TuningConfig(param_grid=nusvc_master_grid, scoring_metric='f1_macro', n_jobs=-1)

# Declaração da configuração de validação
val_config = ValidationConfig(average_method='macro', return_report_dict=False)

incremental_results_master = []

# 2. O Loop Incremental do Doutorando
for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_adicionada = current_features[-1]
    
    print(f"--- Treinando Master Model com {k} Feature(s) [+ {feature_adicionada}] ---")
    
    X_tr_slice = X_train_selected_metrics[current_features]
    X_va_slice = X_val_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        nusvc_base = ClassifierFactory.build(nusvc_config)
        nusvc_champion = ModelOptimizer.optimize(
            estimator=nusvc_base, 
            X_train=X_tr_slice, 
            y_train=y_train_enc, 
            X_val=X_va_slice, 
            y_val=y_val_enc, 
            config=tuning_config_master
        )
        
        metrics_master = ModelEvaluator.evaluate(
            model=nusvc_champion, 
            X_test=X_te_slice, 
            y_test=y_test_enc, 
            config=val_config
        )
        
        # Puxando o F1 Macro com segurança
        f1 = metrics_master.get('f1_macro', metrics_master.get('f1_score', metrics_master.get('f1', 0.0)))
        
        incremental_results_master.append({
            'Dimensao': k,
            'Ultima_Feature_Inclusa': feature_adicionada,
            'F1_Macro': f1
        })
        
    except Exception as e:
        print(f"❌ Falha no passo com {k} features: {e}")
        # Como travamos o nu=0.5, é possível que nas primeiras dimensões (ex: 1 ou 2 features) 
        # a matemática acuse 'infeasible'. O loop apenas pulará para a dimensão seguinte.

In [ ]:
# 3. Tabela de Resultados
df_master_incremental = pd.DataFrame(incremental_results_master)
print("\n📊 Tabela de Desempenho Incremental (Master Model):")
display(df_master_incremental)

# 4. O Gráfico da Vitória
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_master_incremental, 
    x='Dimensao', 
    y='F1_Macro', 
    marker='*', 
    color='gold', # Cor dourada para o modelo campeão
    markeredgecolor='black',
    linewidth=3.0,
    markersize=14
)

plt.title("Impacto da Adição Incremental de Features no Modelo Master (NuSVC Poly Grau 3)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Quantidade de Features Acústicas Utilizadas", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)

# Força o eixo X a exibir os números corretos baseados nas dimensões que rodaram
if not df_master_incremental.empty:
    plt.xticks(df_master_incremental['Dimensao'].tolist())
    
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# Salva o gráfico
if 'REPORTS_DIR' in locals():
    plt.savefig(REPORTS_DIR / f"{DATASET}_E{EXP_NUM}_MasterModel_Incremental.pdf", dpi=300, bbox_inches='tight')

plt.show()

### Busca Exaustiva

In [ ]:
# ==============================================================================
# Busca Exaustiva (Combinatorial) de Features nas Top 10 - SGD
# ==============================================================================
import itertools
import time
import pandas as pd

from rainfall_acoustic_classification.modeling.classifiers import ClassifierFactory, ClassifierConfig
from rainfall_acoustic_classification.modeling.tuning import ModelOptimizer, TuningConfig
from rainfall_acoustic_classification.modeling.validation import ModelEvaluator, ValidationConfig

print("="*60)
print("🚀 Iniciando Busca Exaustiva de Combinações (1023 Possibilidades) - SGD")
print("="*60)

# 1. Pega as Top 10 Features (Certifique-se de que a lista top_10_features já existe)
print(f"Features Base para Combinatória: {top_10_features}\n")

# 2. Configurações super otimizadas do modelo para acelerar o loop
sgd_config = ClassifierConfig(model_type='sgd', random_state=42)
# Vamos usar um grid menor aqui só para a busca rápida não demorar horas
sgd_grid_fast = {'alpha': [0.0001, 0.001], 'penalty': ['l2']} 
tuning_config_fast = TuningConfig(param_grid=sgd_grid_fast, scoring_metric='f1_macro', n_jobs=-1)
val_config = ValidationConfig(average_method='macro', return_report_dict=False)

combinatorial_results = []
total_combinations = 1023 # (2^10 - 1)
contador = 0
start_time = time.time()

# 3. O Loop Exaustivo
# r é o tamanho da combinação (de 1 feature até 10 features)
for r in range(1, len(top_10_features) + 1):
    # Gera todas as combinações de tamanho r
    for subset in itertools.combinations(top_10_features, r):
        subset_list = list(subset)
        contador += 1
        
        # Fatiamento
        X_tr_slice = X_train_selected_metrics[subset_list]
        X_va_slice = X_val_selected_metrics[subset_list]
        X_te_slice = X_test_selected_metrics[subset_list]
        
        try:
            # Treino e Avaliação
            sgd_base = ClassifierFactory.build(sgd_config)
            sgd_champion = ModelOptimizer.optimize(
                estimator=sgd_base, X_train=X_tr_slice, y_train=y_train_enc, 
                X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_fast
            )
            metrics_sgd = ModelEvaluator.evaluate(model=sgd_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
            
            f1 = metrics_sgd.get('f1_macro', metrics_sgd.get('f1_score', 0.0))
            
            combinatorial_results.append({
                'Tamanho': r,
                'F1_Macro': f1,
                'Features': " + ".join(subset_list)
            })
            
        except Exception as e:
            continue
            
        # Log de progresso a cada 100 iterações
        if contador % 100 == 0:
            print(f"Progresso: {contador}/{total_combinations} testadas... Melhor F1 até agora: {max([res['F1_Macro'] for res in combinatorial_results]):.4f}")

elapsed = time.time() - start_time
print(f"\n✅ Busca concluída em {elapsed/60:.2f} minutos!")

# 4. Tabela de Ouro (As 10 melhores combinações do Universo)
df_combinatorial = pd.DataFrame(combinatorial_results)
df_combinatorial = df_combinatorial.sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)

print("\n🏆 AS 10 MELHORES COMBINAÇÕES ABSOLUTAS:")
display(df_combinatorial.head(10))

# (Opcional) Salva para o artigo
if 'REPORTS_DIR' in locals():
    df_combinatorial.to_csv(REPORTS_DIR / f"{DATASET}_E{EXP_NUM}_Exhaustive_Search_SGD.csv", index=False)

In [ ]:
# ==============================================================================
# Busca Exaustiva de Features (1023 Possibilidades) - Regressão Logística (LR)
# ==============================================================================
import itertools
import time
import pandas as pd
from rainfall_acoustic_classification.modeling.classifiers import ClassifierFactory, ClassifierConfig
from rainfall_acoustic_classification.modeling.tuning import ModelOptimizer, TuningConfig
from rainfall_acoustic_classification.modeling.validation import ModelEvaluator, ValidationConfig

print("="*60)
print("🚀 Iniciando Busca Exaustiva - Regressão Logística (LR)")
print("="*60)

# Grade travada para velocidade máxima de busca
lr_config = ClassifierConfig(model_type='lr', random_state=42)
lr_grid_fast = {'C': [1.0], 'class_weight': ['balanced'], 'solver': ['lbfgs'], 'max_iter': [1000]} 
tuning_config_lr_fast = TuningConfig(param_grid=lr_grid_fast, scoring_metric='f1_macro', n_jobs=-1)
val_config = ValidationConfig(average_method='macro', return_report_dict=False)

combinatorial_results_lr = []
contador = 0
start_time = time.time()

for r in range(1, len(top_10_features) + 1):
    for subset in itertools.combinations(top_10_features, r):
        subset_list = list(subset)
        contador += 1
        
        X_tr_slice = X_train_selected_metrics[subset_list]
        X_va_slice = X_val_selected_metrics[subset_list]
        X_te_slice = X_test_selected_metrics[subset_list]
        
        try:
            lr_base = ClassifierFactory.build(lr_config)
            lr_champion = ModelOptimizer.optimize(
                estimator=lr_base, X_train=X_tr_slice, y_train=y_train_enc, 
                X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_lr_fast
            )
            metrics_lr = ModelEvaluator.evaluate(model=lr_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
            
            f1 = metrics_lr.get('f1_macro', metrics_lr.get('f1_score', 0.0))
            combinatorial_results_lr.append({'Tamanho': r, 'F1_Macro': f1, 'Features': " + ".join(subset_list)})
        except:
            continue
            
        if contador % 100 == 0:
            print(f"Progresso LR: {contador}/1023... Melhor F1 até agora: {max([res['F1_Macro'] for res in combinatorial_results_lr]):.4f}")

elapsed = time.time() - start_time
print(f"\n✅ Busca LR concluída em {elapsed/60:.2f} minutos!")

df_comb_lr = pd.DataFrame(combinatorial_results_lr).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
print("\n🏆 AS 5 MELHORES COMBINAÇÕES (LR):")
display(df_comb_lr.head(5))

if 'REPORTS_DIR' in locals():
    df_comb_lr.to_csv(REPORTS_DIR / f"{DATASET}_E{EXP_NUM}_Exhaustive_Search_LR.csv", index=False)

In [ ]:
# ==============================================================================
# Busca Exaustiva de Features (1023 Possibilidades) - Random Forest (RF)
# ==============================================================================
print("="*60)
print("🚀 Iniciando Busca Exaustiva - Random Forest (RF) [PODE DEMORAR]")
print("="*60)

# Grade travada em 100 árvores para economizar tempo (do contrário demoraria dias)
rf_config = ClassifierConfig(model_type='rf', random_state=42)
rf_grid_fast = {'n_estimators': [100], 'max_depth': [None], 'min_samples_leaf': [2]} 
tuning_config_rf_fast = TuningConfig(param_grid=rf_grid_fast, scoring_metric='f1_macro', n_jobs=-1)

combinatorial_results_rf = []
contador = 0
start_time = time.time()

for r in range(1, len(top_10_features) + 1):
    for subset in itertools.combinations(top_10_features, r):
        subset_list = list(subset)
        contador += 1
        
        X_tr_slice = X_train_selected_metrics[subset_list]
        X_va_slice = X_val_selected_metrics[subset_list]
        X_te_slice = X_test_selected_metrics[subset_list]
        
        try:
            rf_base = ClassifierFactory.build(rf_config)
            rf_champion = ModelOptimizer.optimize(
                estimator=rf_base, X_train=X_tr_slice, y_train=y_train_enc, 
                X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_rf_fast
            )
            metrics_rf = ModelEvaluator.evaluate(model=rf_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
            
            f1 = metrics_rf.get('f1_macro', metrics_rf.get('f1_score', 0.0))
            combinatorial_results_rf.append({'Tamanho': r, 'F1_Macro': f1, 'Features': " + ".join(subset_list)})
        except:
            continue
            
        if contador % 100 == 0:
            print(f"Progresso RF: {contador}/1023... Melhor F1 até agora: {max([res['F1_Macro'] for res in combinatorial_results_rf]):.4f}")

elapsed = time.time() - start_time
print(f"\n✅ Busca RF concluída em {elapsed/60:.2f} minutos!")

df_comb_rf = pd.DataFrame(combinatorial_results_rf).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
print("\n🏆 AS 5 MELHORES COMBINAÇÕES (RF):")
display(df_comb_rf.head(5))

if 'REPORTS_DIR' in locals():
    df_comb_rf.to_csv(REPORTS_DIR / f"{DATASET}_E{EXP_NUM}_Exhaustive_Search_RF.csv", index=False)

In [ ]:
# ==============================================================================
# Busca Exaustiva de Features (1023 Possibilidades) - XGBoost (XGB)
# ==============================================================================
print("="*60)
print("🚀 Iniciando Busca Exaustiva - XGBoost (XGB) [PODE DEMORAR]")
print("="*60)

# Grade travada numa taxa de aprendizado agressiva para buscar convergência rápida
xgb_config = ClassifierConfig(model_type='xgb', random_state=42)
xgb_grid_fast = {'n_estimators': [100], 'max_depth': [3], 'learning_rate': [0.1]} 
tuning_config_xgb_fast = TuningConfig(param_grid=xgb_grid_fast, scoring_metric='f1_macro', n_jobs=-1)

combinatorial_results_xgb = []
contador = 0
start_time = time.time()

for r in range(1, len(top_10_features) + 1):
    for subset in itertools.combinations(top_10_features, r):
        subset_list = list(subset)
        contador += 1
        
        X_tr_slice = X_train_selected_metrics[subset_list]
        X_va_slice = X_val_selected_metrics[subset_list]
        X_te_slice = X_test_selected_metrics[subset_list]
        
        try:
            xgb_base = ClassifierFactory.build(xgb_config)
            xgb_champion = ModelOptimizer.optimize(
                estimator=xgb_base, X_train=X_tr_slice, y_train=y_train_enc, 
                X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_xgb_fast
            )
            metrics_xgb = ModelEvaluator.evaluate(model=xgb_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
            
            f1 = metrics_xgb.get('f1_macro', metrics_xgb.get('f1_score', 0.0))
            combinatorial_results_xgb.append({'Tamanho': r, 'F1_Macro': f1, 'Features': " + ".join(subset_list)})
        except:
            continue
            
        if contador % 100 == 0:
            print(f"Progresso XGB: {contador}/1023... Melhor F1 até agora: {max([res['F1_Macro'] for res in combinatorial_results_xgb]):.4f}")

elapsed = time.time() - start_time
print(f"\n✅ Busca XGB concluída em {elapsed/60:.2f} minutos!")

df_comb_xgb = pd.DataFrame(combinatorial_results_xgb).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
print("\n🏆 AS 5 MELHORES COMBINAÇÕES (XGB):")
display(df_comb_xgb.head(5))

if 'REPORTS_DIR' in locals():
    df_comb_xgb.to_csv(REPORTS_DIR / f"{DATASET}_E{EXP_NUM}_Exhaustive_Search_XGB.csv", index=False)

In [ ]:
# ==============================================================================
# Busca Exaustiva ULTRA-OTIMIZADA - MASTER MODEL (NuSVC Poly Grau 3)
# ==============================================================================
import itertools
import time
import pandas as pd
from sklearn.svm import NuSVC
from rainfall_acoustic_classification.modeling.validation import ModelEvaluator, ValidationConfig

print("="*60)
print("🚀 Iniciando Busca Exaustiva (Bare-Metal Otimizado) - NuSVC Poly")
print("="*60)

# 1. Poda do Espaço de Busca (Ignora combinações com menos de 6 features)
MIN_FEATURES = 6
MAX_FEATURES = len(top_10_features)

# Validador inviolável
val_config = ValidationConfig(average_method='macro', return_report_dict=False)

combinatorial_results_master = []
contador = 0
start_time = time.time()

# Calcula o número real de combinações que faremos (Soma de C(10,6) até C(10,10))
total_combinations_pruned = sum([len(list(itertools.combinations(top_10_features, r))) for r in range(MIN_FEATURES, MAX_FEATURES + 1)])
print(f"📉 Poda Ativada: Testando apenas subconjuntos de {MIN_FEATURES} a {MAX_FEATURES} features.")
print(f"📉 Combinações reduzidas de 1023 para apenas {total_combinations_pruned} testes focados!\n")

for r in range(MIN_FEATURES, MAX_FEATURES + 1):
    for subset in itertools.combinations(top_10_features, r):
        subset_list = list(subset)
        contador += 1
        
        # Fatiamento
        X_tr_slice = X_train_selected_metrics[subset_list]
        X_te_slice = X_test_selected_metrics[subset_list]
        
        try:
            # 2. Bypass do Otimizador: Instanciação e Treino Direto (Máxima Velocidade)
            clf_master = NuSVC(
                kernel='poly', degree=3, coef0=1.0, nu=0.5, 
                gamma='scale', class_weight='balanced', 
                probability=True, random_state=42
            )
            
            # Treino direto no Train (Sem overhead de validação cruzada)
            clf_master.fit(X_tr_slice, y_train_enc)
            
            # 3. Avaliação Direta no Teste
            metrics_master = ModelEvaluator.evaluate(
                model=clf_master, X_test=X_te_slice, y_test=y_test_enc, config=val_config
            )
            
            f1 = metrics_master.get('f1_macro', metrics_master.get('f1_score', 0.0))
            
            combinatorial_results_master.append({
                'Tamanho': r, 
                'F1_Macro': f1, 
                'Features': " + ".join(subset_list)
            })
            
        except Exception as e:
            # Continua silenciosamente se houver qualquer anomalia matemática residual
            continue
            
        if contador % 50 == 0:
            melhor_f1_atual = max([res['F1_Macro'] for res in combinatorial_results_master]) if combinatorial_results_master else 0.0
            print(f"Progresso: {contador}/{total_combinations_pruned}... Melhor F1: {melhor_f1_atual:.4f}")

elapsed = time.time() - start_time
print(f"\n✅ Busca Exaustiva Bare-Metal concluída em apenas {elapsed/60:.2f} minutos!")

# Transformando e Ordenando Resultados
df_comb_master = pd.DataFrame(combinatorial_results_master)

if not df_comb_master.empty:
    df_comb_master = df_comb_master.sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
    print(f"\n🏆 AS 10 MELHORES COMBINAÇÕES ABSOLUTAS ({DATASET}):")
    display(df_comb_master.head(10))

    if 'REPORTS_DIR' in locals():
        df_comb_master.to_csv(REPORTS_DIR / f"{DATASET}_E{EXP_NUM}_Exhaustive_Search_MasterModel.csv", index=False)